# M5A5 - Segmentação de Falhas em Tecidos

Na prática de hoje vamos refinar um modelo para a tarefa de segmentação de falhas em tecidos.

Esse notebook está estruturado da seguinte forma.

- Introdução
- Carregar Base de Dados
- Refinar Modelo
- Validar Modelo
- Próximos passos
- Atividade Complementares

## Introdução

Instalação para os que ainda não possuem a biblioteca instalada.

In [3]:
!pip install torch torchvision tqdm ipywidgets

Importar as bibliotecas

In [4]:
import torch
import torchvision
from tqdm.notebook import tqdm


## Carregar Base de Dados

A primeira tarefa para refinar um modelo é criar a base de dados.

Referência: https://docs.pytorch.org/vision/main/generated/torchvision.datasets.DTD.html

In [5]:
# Define transforms: Resize and normalize images as expected by most pre-trained models (e.g., ImageNet models use 224x224 input).
# Data augmentation is also crucial for better performance.
image_transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224), # Common input size for ImageNet models
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the dataset
# Set download=True to automatically fetch the dataset if not already present
train_dataset = torchvision.datasets.DTD(root="./data", split="train", download=True, transform=image_transforms)
val_dataset = torchvision.datasets.DTD(root="./data", split="val", download=True, transform=image_transforms)
test_dataset = torchvision.datasets.DTD(root="./data", split="test", download=True, transform=image_transforms)

# Define DataLoaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
_ = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

100%|██████████| 625M/625M [00:57<00:00, 10.9MB/s]  


## Refinar Modelo

Na prática de hoje iremos refinar o modelo **ResNets** disponível no torchvision.

In [6]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)

num_classes = 47

# Get the number of input features for the final layer
in_features = model.fc.in_features

# Replace the existing fully connected layer with a new one
model.fc = torch.nn.Linear(in_features, num_classes)

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Treinamento do modelo
model.train()
epochs = 100 # Alterar para treinar mais epocas.
for epoch in tqdm(range(epochs)):
    iteration = 0
    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if iteration % 100 == 0:
            print(f"Total loss: {loss.item()}")
        iteration += 1



Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\kaiog/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 42.1MB/s]


  0%|          | 0/100 [00:00<?, ?it/s]

Total loss: 4.187209129333496
Total loss: 2.686765193939209
Total loss: 1.8354071378707886
Total loss: 1.348679780960083
Total loss: 1.105711579322815
Total loss: 0.9329673051834106
Total loss: 0.5679405927658081
Total loss: 0.424856573343277
Total loss: 0.5207191109657288
Total loss: 0.18962109088897705
Total loss: 0.3620726466178894
Total loss: 0.17596063017845154
Total loss: 0.15490397810935974
Total loss: 0.287822961807251
Total loss: 0.1828949898481369
Total loss: 0.16291680932044983
Total loss: 0.057243168354034424
Total loss: 0.06856324523687363
Total loss: 0.09129845350980759
Total loss: 0.06362097710371017
Total loss: 0.07443635910749435
Total loss: 0.039947301149368286
Total loss: 0.04319363832473755
Total loss: 0.03496381267905235
Total loss: 0.06662935763597488
Total loss: 0.032728955149650574
Total loss: 0.04672364145517349
Total loss: 0.035247936844825745
Total loss: 0.08458544313907623
Total loss: 0.06226836144924164
Total loss: 0.020630011335015297
Total loss: 0.0487534

## Validar Modelo

Agora vamos testar o nosso modelo.

In [7]:
num_correct = 0
num_samples = 0
    
# Set model to evaluation mode
model.eval()

# Disable gradient calculation for inference
with torch.no_grad():
    for images, labels in test_loader:
            x = images.to(device)
            y = labels.to(device)

            scores = model(x)

            # Get the index (class) with the highest score
            _, predictions = scores.max(1)
            
            # Count correct predictions
            num_correct += (predictions == y).sum().item()
            # Count total samples
            num_samples += predictions.size(0)

    # Calculate and print accuracy
    accuracy = float(num_correct) / float(num_samples) * 100
    print(f'Got {num_correct} / {num_samples} with accuracy {accuracy:.2f}%')


Got 1223 / 1880 with accuracy 65.05%


## Próximos Passos e Referências

Nas próximas práticas vamos continuar trabalhando com problemas reais que envolvem Visão Computacional.

Uma lista não exaustiva de referências segue:

- https://docs.pytorch.org/vision/main/generated/torchvision.datasets.DTD.html
- https://docs.pytorch.org/vision/main/models/generated/torchvision.models.resnet18.html
- https://huggingface.co/datasets
- https://pytorch.org/
- https://docs.pytorch.org/vision/main/models.html
- https://opencv.org/
- https://learnopencv.com/blogs/
- https://pyimagesearch.com/

## Atividades Complementares (Opcional)

- [ ] Tente por menos tempo faz com que a perfomance do modelo matenha-se a mesma?
- [ ] Tente alterar alguns hiperparâmetros de treinamento, batch e learning rate e veja como isso altera os resultados.

In [8]:
# Define transforms: Resize and normalize images as expected by most pre-trained models (e.g., ImageNet models use 224x224 input).
# Data augmentation is also crucial for better performance.
image_transforms = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224), # Common input size for ImageNet models
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load the dataset
# Set download=True to automatically fetch the dataset if not already present
train_dataset = torchvision.datasets.DTD(root="./data", split="train", download=True, transform=image_transforms)
val_dataset = torchvision.datasets.DTD(root="./data", split="val", download=True, transform=image_transforms)
test_dataset = torchvision.datasets.DTD(root="./data", split="test", download=True, transform=image_transforms)

# Define DataLoaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size = 64, shuffle=True)
_ = torch.utils.data.DataLoader(val_dataset, batch_size = 64, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1, shuffle=False)

In [9]:
model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)

num_classes = 47

# Get the number of input features for the final layer
in_features = model.fc.in_features

# Replace the existing fully connected layer with a new one
model.fc = torch.nn.Linear(in_features, num_classes)

# Move the model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()

# Observe that all parameters are being optimized
#optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

optimizer = torch.optim.SGD(model.parameters(), lr=0.0005, momentum=0.9, weight_decay=1e-4)

# Treinamento do modelo
model.train()
epochs = 100 # Alterar para treinar mais epocas.
for epoch in tqdm(range(epochs)):
    iteration = 0
    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if iteration % 100 == 0:
            print(f"Total loss: {loss.item()}")
        iteration += 1



  0%|          | 0/100 [00:00<?, ?it/s]

Total loss: 4.19838809967041
Total loss: 3.613870143890381
Total loss: 3.060701847076416
Total loss: 2.754345655441284
Total loss: 2.301788806915283
Total loss: 1.8863247632980347
Total loss: 2.0039689540863037
Total loss: 1.564668893814087
Total loss: 1.2813713550567627
Total loss: 1.3113470077514648
Total loss: 1.1227457523345947
Total loss: 1.2238926887512207
Total loss: 1.0878381729125977
Total loss: 0.8793790340423584
Total loss: 0.6727840900421143
Total loss: 0.8053810596466064
Total loss: 0.5446638464927673
Total loss: 0.5207352042198181
Total loss: 0.63743656873703
Total loss: 0.5326442718505859
Total loss: 0.5718475580215454
Total loss: 0.6034448146820068
Total loss: 0.5554599761962891
Total loss: 0.45876288414001465
Total loss: 0.3590071499347687
Total loss: 0.3041892945766449
Total loss: 0.3324591815471649
Total loss: 0.31781113147735596
Total loss: 0.32521355152130127
Total loss: 0.20256374776363373
Total loss: 0.2770799994468689
Total loss: 0.2121581882238388
Total loss: 0

In [ ]:
num_correct = 0
num_samples = 0
    
# Set model to evaluation mode
model.eval()

# Disable gradient calculation for inference
with torch.no_grad():
    for images, labels in test_loader:
            x = images.to(device)
            y = labels.to(device)

            scores = model(x)

            # Get the index (class) with the highest score
            _, predictions = scores.max(1)
            
            # Count correct predictions
            num_correct += (predictions == y).sum().item()
            # Count total samples
            num_samples += predictions.size(0)

    # Calculate and print accuracy
    accuracy = float(num_correct) / float(num_samples) * 100
    print(f'Got {num_correct} / {num_samples} with accuracy {accuracy:.2f}%')
